Use the repository-root [RUNBOOK](../docs/RUNBOOK.md) for schema-3 reproduction and [verification](../docs/FLIGHTTIME_VERIFICATION.md) for current acceptance. Earlier metrics and retired experiment commands are historical; see the [archive](../evidence/archive/pre-consolidation/INDEX.md). M9 has no new benchmark claim.

# Spark Standalone smoke test

This notebook checks the small Dockerized Spark Standalone cluster used by the Airline project. The master is supplied by the runtime environment, so the notebook deliberately does not hardcode a master URL.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("AirlineSparkTest").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Spark master: {spark.sparkContext.master}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")

## Tiny distributed job

The range is split into eight partitions. The aggregation introduces a shuffle, and `collect()` triggers execution. A successful result verifies the path from the notebook driver to the standalone master, then to the workers and their executors/tasks, and back to the driver.

In [ ]:
numbers = spark.range(0, 1000, numPartitions=8)
summary = (
    numbers
    .withColumn("bucket", F.col("id") % 4)
    .groupBy("bucket")
    .agg(F.count("*").alias("rows"), F.sum("id").alias("id_sum"))
    .orderBy("bucket")
)

result = summary.collect()
print(f"Input partitions: {numbers.rdd.getNumPartitions()}")
for row in result:
    print(row)

assert sum(row.rows for row in result) == 1000
assert len(result) == 4
print("Distributed Spark job completed successfully.")
spark.stop()
print("Spark session stopped; cluster resources released.")